# Lezione 3 — Validazione, overfitting e tuning

**Obiettivo:** capire perché non basta addestrare un modello; bisogna anche validarlo bene e scegliere gli iperparametri.

## Dataset
Useriamo il dataset **Breast Cancer Wisconsin** di `scikit-learn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 1) Caricamento dati

In [ ]:
bc = load_breast_cancer(as_frame=True)
df = bc.frame.copy()
df["target_name"] = df["target"].map(dict(enumerate(bc.target_names)))
df.head()

In [ ]:
df["target_name"].value_counts()

## 2) Definizione di X e y

In [ ]:
X = df.drop(columns=["target", "target_name"])
y = df["target_name"]

## 3) Train/Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

## 4) Decision tree: esempio di overfitting

In [ ]:
tree_full = DecisionTreeClassifier(max_depth=None, random_state=42)
tree_full.fit(X_train, y_train)

train_acc_full = tree_full.score(X_train, y_train)
test_acc_full = tree_full.score(X_test, y_test)

train_acc_full, test_acc_full

In [ ]:
tree_small = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_small.fit(X_train, y_train)

train_acc_small = tree_small.score(X_train, y_train)
test_acc_small = tree_small.score(X_test, y_test)

train_acc_small, test_acc_small

In [ ]:
pd.DataFrame({
    "model": ["Tree full", "Tree max_depth=3"],
    "train_acc": [train_acc_full, train_acc_small],
    "test_acc": [test_acc_full, test_acc_small]
})

### Interpretazione
Se la accuracy sul train è molto maggiore di quella sul test, il modello sta probabilmente **overfittando**.

## 5) Cross-validation con kNN

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
cv_scores = cross_val_score(knn, X_train_s, y_train, cv=5)
cv_scores, cv_scores.mean(), cv_scores.std()

## 6) Grid Search sugli iperparametri

In [ ]:
param_grid = {
    "n_neighbors": [1, 3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"]
}

grid = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    cv=5,
    n_jobs=-1
)

grid.fit(X_train_s, y_train)
grid.best_params_, grid.best_score_

In [ ]:
best_knn = grid.best_estimator_
pred_best = best_knn.predict(X_test_s)

acc_best = accuracy_score(y_test, pred_best)
acc_best

In [ ]:
print(classification_report(y_test, pred_best))

In [ ]:
cm = confusion_matrix(y_test, pred_best, labels=best_knn.classes_)
pd.DataFrame(cm, index=[f"true_{c}" for c in best_knn.classes_], columns=[f"pred_{c}" for c in best_knn.classes_])

## 7) Comunicazione dei risultati

Scrivi una breve relazione:
1. Quale modello overfitta di più?
2. Perché la cross-validation è utile?
3. Quali iperparametri ha scelto la grid search?
4. Useresti il modello finale in produzione? Perché?